# Optimizing VS Code for Large Workspaces

This notebook provides strategies and tools for handling large workspaces in VS Code, with a focus on Python projects and Pylance optimization. We'll cover techniques to analyze workspace size, configure Pylance settings, and create custom workspace configurations for better performance.

## Contents

1. Setting Up the Environment
2. Analyzing Workspace Size
3. Performance Optimization Techniques
4. Testing Different Configuration Settings
5. Creating a Custom Workspace Scanner

## 1. Setting Up the Environment

First, let's import necessary libraries to analyze and work with large directory structures.

In [ ]:
import os
import sys
import json
import time
import logging
import pathlib
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import re
import multiprocessing as mp
from tqdm.notebook import tqdm

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('workspace-analyzer')

# Create a timer context manager to measure performance
class Timer:
    def __init__(self, name):
        self.name = name
    
    def __enter__(self):
        self.start = time.time()
        return self
    
    def __exit__(self, *args):
        self.end = time.time()
        self.interval = self.end - self.start
        logger.info(f"{self.name} took {self.interval:.2f} seconds")
        return self.interval

## 2. Analyzing Workspace Size

Let's create functions to analyze workspace size, count files by type, and identify potential performance bottlenecks.

In [ ]:
def scan_workspace(workspace_path, exclude_patterns=None):
    """
    Scan a workspace and gather statistics about file types and sizes
    
    Args:
        workspace_path: Path to the workspace root
        exclude_patterns: List of regex patterns to exclude
        
    Returns:
        Dictionary with workspace statistics
    """
    if exclude_patterns is None:
        exclude_patterns = [
            r'\.git',
            r'node_modules',
            r'__pycache__',
            r'\.venv',
            r'env',
            r'venv',
            r'dist',
            r'build'
        ]
    
    # Compile exclude patterns
    exclude_regex = [re.compile(pattern) for pattern in exclude_patterns]
    
    stats = {
        'total_files': 0,
        'total_dirs': 0,
        'file_types': Counter(),
        'file_sizes': [],
        'largest_files': [],
        'deepest_paths': [],
        'files_by_dir': defaultdict(int)
    }
    
    max_files_to_track = 100  # Track the top N largest files
    
    logger.info(f"Scanning workspace: {workspace_path}")
    start_time = time.time()
    
    # Walk the directory
    for root, dirs, files in os.walk(workspace_path):
        # Check if this directory should be excluded
        rel_root = os.path.relpath(root, workspace_path)
        if any(pattern.search(rel_root) for pattern in exclude_regex):
            # Skip this directory and all subdirectories
            dirs[:] = []
            continue
            
        stats['total_dirs'] += 1
        
        for file in files:
            file_path = os.path.join(root, file)
            try:
                # Get file extension (lowercase)
                _, ext = os.path.splitext(file)
                ext = ext.lower()
                
                # Get file size
                file_size = os.path.getsize(file_path)
                
                # Update statistics
                stats['total_files'] += 1
                stats['file_types'][ext] += 1
                stats['file_sizes'].append(file_size)
                
                # Track largest files
                if len(stats['largest_files']) < max_files_to_track:
                    stats['largest_files'].append((file_path, file_size))
                    stats['largest_files'].sort(key=lambda x: x[1], reverse=True)
                elif file_size > stats['largest_files'][-1][1]:
                    stats['largest_files'].pop()
                    stats['largest_files'].append((file_path, file_size))
                    stats['largest_files'].sort(key=lambda x: x[1], reverse=True)
                
                # Track directory depth
                path_depth = len(Path(rel_root).parts)
                if path_depth > 0:  # Ignore root
                    stats['deepest_paths'].append((rel_root, path_depth))
                    stats['deepest_paths'].sort(key=lambda x: x[1], reverse=True)
                    if len(stats['deepest_paths']) > max_files_to_track:
                        stats['deepest_paths'].pop()
                
                # Count files by directory
                parent_dir = os.path.dirname(file_path)
                stats['files_by_dir'][parent_dir] += 1
                
            except (PermissionError, FileNotFoundError) as e:
                logger.warning(f"Error processing file {file_path}: {e}")
    
    end_time = time.time()
    stats['scan_time'] = end_time - start_time
    
    return stats

# Function to visualize workspace statistics
def visualize_workspace_stats(stats):
    """Generate visualizations for workspace statistics"""
    # Create a figure with subplots
    fig, axs = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. File Types Distribution
    file_types = stats['file_types']
    # Get top file types
    top_types = dict(sorted(file_types.items(), key=lambda x: x[1], reverse=True)[:15])
    
    axs[0, 0].bar(top_types.keys(), top_types.values())
    axs[0, 0].set_title('Top File Extensions')
    axs[0, 0].set_xlabel('Extension')
    axs[0, 0].set_ylabel('Count')
    axs[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. File Size Distribution (log scale histogram)
    file_sizes = stats['file_sizes']
    if file_sizes:
        axs[0, 1].hist(file_sizes, bins=50)
        axs[0, 1].set_title('File Size Distribution')
        axs[0, 1].set_xlabel('File Size (bytes)')
        axs[0, 1].set_ylabel('Count')
        axs[0, 1].set_xscale('log')
    
    # 3. Directory Depth Analysis
    depths = [d[1] for d in stats['deepest_paths']]
    if depths:
        axs[1, 0].hist(depths, bins=range(min(depths), max(depths) + 2))
        axs[1, 0].set_title('Directory Depth Distribution')
        axs[1, 0].set_xlabel('Directory Depth')
        axs[1, 0].set_ylabel('Count')
    
    # 4. Files per Directory (top directories)
    dirs_by_file_count = sorted(stats['files_by_dir'].items(), key=lambda x: x[1], reverse=True)[:10]
    dir_names = [os.path.basename(d[0]) for d in dirs_by_file_count]
    dir_counts = [d[1] for d in dirs_by_file_count]
    
    axs[1, 1].bar(dir_names, dir_counts)
    axs[1, 1].set_title('Top Directories by File Count')
    axs[1, 1].set_xlabel('Directory')
    axs[1, 1].set_ylabel('Number of Files')
    axs[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"Total Files: {stats['total_files']}")
    print(f"Total Directories: {stats['total_dirs']}")
    print(f"Scan Time: {stats['scan_time']:.2f} seconds")
    
    # Print largest files
    print("\nLargest Files:")
    for path, size in stats['largest_files'][:10]:
        print(f"{path}: {size/1024/1024:.2f} MB")
    
    # Print deepest paths
    print("\nDeepest Paths:")
    for path, depth in stats['deepest_paths'][:10]:
        print(f"{path}: {depth} levels deep")

In [ ]:
# Example usage on a sample workspace
# To analyze your actual workspace, replace this with your workspace path
WORKSPACE_PATH = "/workspaces/windgapacademy"

# Let's run our workspace scanner with performance timing
with Timer("Workspace Scan"):
    workspace_stats = scan_workspace(WORKSPACE_PATH)

# Visualize the results
visualize_workspace_stats(workspace_stats)

## 3. Performance Optimization Techniques

Now let's implement different techniques for optimizing workspace loading, including path filtering, multiprocessing for file enumeration, and caching mechanisms.

In [ ]:
# Helper functions for performance optimization
def is_excluded(path, exclude_patterns):
    """Check if a path matches any exclude pattern"""
    for pattern in exclude_patterns:
        if re.search(pattern, path):
            return True
    return False

def scan_directory_chunk(args):
    """Process a chunk of directories for multiprocessing"""
    start_dir, exclude_patterns = args
    file_count = 0
    file_list = []
    
    for root, dirs, files in os.walk(start_dir):
        # Filter out excluded directories
        rel_path = os.path.relpath(root, WORKSPACE_PATH)
        if is_excluded(rel_path, exclude_patterns):
            dirs[:] = []  # Skip subdirectories
            continue
            
        for file in files:
            file_path = os.path.join(root, file)
            rel_file_path = os.path.relpath(file_path, WORKSPACE_PATH)
            
            if not is_excluded(rel_file_path, exclude_patterns):
                file_count += 1
                file_list.append(rel_file_path)
    
    return file_count, file_list

# Function to scan workspace with multiprocessing
def parallel_scan_workspace(workspace_path, exclude_patterns=None, num_processes=None):
    """
    Scan a workspace using multiprocessing for better performance
    
    Args:
        workspace_path: Path to the workspace root
        exclude_patterns: List of regex patterns to exclude
        num_processes: Number of processes to use (defaults to CPU count)
        
    Returns:
        Total file count and list of files
    """
    if exclude_patterns is None:
        exclude_patterns = [
            r'\.git',
            r'node_modules',
            r'__pycache__',
            r'\.venv',
            r'env',
            r'venv',
            r'dist',
            r'build'
        ]
    
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    # Get top-level directories to distribute work
    top_dirs = []
    for item in os.listdir(workspace_path):
        item_path = os.path.join(workspace_path, item)
        if os.path.isdir(item_path) and not is_excluded(item, exclude_patterns):
            top_dirs.append(item_path)
    
    # If there are fewer top directories than processes, adjust process count
    num_processes = min(num_processes, len(top_dirs))
    if num_processes == 0:
        num_processes = 1
        top_dirs = [workspace_path]
    
    logger.info(f"Scanning workspace with {num_processes} processes")
    
    # Create process pool and distribute work
    with mp.Pool(processes=num_processes) as pool:
        args = [(d, exclude_patterns) for d in top_dirs]
        results = list(tqdm(pool.imap(scan_directory_chunk, args), total=len(args)))
    
    # Combine results
    total_files = sum(r[0] for r in results)
    all_files = [f for r in results for f in r[1]]
    
    return total_files, all_files

# Function to generate optimal VS Code settings based on workspace analysis
def generate_vscode_settings(workspace_path, scan_results):
    """
    Generate optimized VS Code settings based on workspace analysis
    
    Args:
        workspace_path: Path to the workspace root
        scan_results: Results from workspace scan
        
    Returns:
        Dictionary with VS Code settings
    """
    # Extract file types for targeted exclusion
    file_types = scan_results['file_types']
    file_count_by_dir = scan_results['files_by_dir']
    
    # Find directories with large numbers of files
    large_dirs = sorted(
        [(os.path.relpath(d, workspace_path), c) for d, c in file_count_by_dir.items()],
        key=lambda x: x[1], 
        reverse=True
    )[:20]
    
    # Find Python files (for include patterns)
    python_dirs = defaultdict(int)
    
    for root, dirs, files in os.walk(workspace_path):
        rel_root = os.path.relpath(root, workspace_path)
        
        python_files = [f for f in files if f.endswith('.py')]
        if python_files:
            # Count up the directory tree
            path_parts = Path(rel_root).parts
            for i in range(len(path_parts)):
                subpath = os.path.join(*path_parts[:i+1]) if i > 0 else path_parts[0]
                python_dirs[subpath] += len(python_files)
    
    # Sort Python directories by file count
    top_python_dirs = sorted(python_dirs.items(), key=lambda x: x[1], reverse=True)
    
    # Create include patterns for directories with Python files
    include_patterns = []
    for dir_path, count in top_python_dirs[:10]:  # Top 10 Python directories
        if count > 5:  # Only include directories with more than 5 Python files
            include_patterns.append(f"{dir_path}/**/*.py")
    
    # Generate exclude patterns for large non-Python directories
    exclude_patterns = []
    for dir_path, count in large_dirs:
        # Skip if this is a primary Python directory
        if any(dir_path.startswith(p.split('/**')[0]) for p in include_patterns):
            continue
        exclude_patterns.append(f"{dir_path}/**")
    
    # Generate VS Code settings
    vscode_settings = {
        "python.analysis.diagnosticMode": "openFilesOnly",
        "python.analysis.typeCheckingMode": "basic",
        "python.analysis.autoSearchPaths": True,
        "python.analysis.extraPaths": [d.split('/**')[0] for d in include_patterns[:5]],
        "python.analysis.include": include_patterns,
        "python.analysis.exclude": [
            "**/node_modules/**",
            "**/__pycache__/**",
            "**/build/**",
            ".git/**",
            "**/.venv/**",
            "**/env/**",
            "dist/**"
        ] + exclude_patterns,
        "python.analysis.indexing": False,
        "python.analysis.packageIndexDepths": [{"name": "", "depth": 1}],
        "python.analysis.autoImportCompletions": False,
        "python.analysis.inlayHints.functionReturnTypes": False,
        "python.analysis.inlayHints.variableTypes": False,
        "python.languageServer": "Pylance",
        "python.analysis.memory.keepLibraryAst": False
    }
    
    return vscode_settings

# Function to save settings to a file
def save_vscode_settings(settings, output_path=".vscode/settings.json"):
    """Save VS Code settings to a file"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with open(output_path, 'w') as f:
        json.dump(settings, f, indent=2)
    
    logger.info(f"Saved VS Code settings to {output_path}")

In [ ]:
# Let's run our parallel scanner to see the performance difference
with Timer("Parallel Workspace Scan"):
    total_files, file_list = parallel_scan_workspace(WORKSPACE_PATH)

print(f"Total files found: {total_files}")
print(f"First 10 files: {file_list[:10]}")

# Generate optimized VS Code settings
optimized_settings = generate_vscode_settings(WORKSPACE_PATH, workspace_stats)

# Display the optimized settings
print("\nOptimized VS Code Settings:")
print(json.dumps(optimized_settings, indent=2))

## 4. Testing Different Configuration Settings

Now let's create and test various VS Code configuration settings using Python dictionaries. We'll measure the performance impacts of different include/exclude patterns and extraPaths configurations.

In [ ]:
# Define different configuration strategies to test
def get_test_configurations():
    """Generate different VS Code configurations to test"""
    return {
        "minimal": {
            "python.analysis.diagnosticMode": "openFilesOnly",
            "python.analysis.autoSearchPaths": False,
            "python.analysis.indexing": False,
            "python.analysis.exclude": [
                "**/node_modules/**",
                "**/__pycache__/**",
                "**/build/**",
                ".git/**",
                "**/.venv/**",
                "**/env/**",
                "dist/**",
                "o3de/**",
                "unity-integration/**",
                "unity-setup/**",
                "test-results/**",
                "cypress/**",
                "playwright/**"
            ]
        },
        "light_mode": {
            "python.analysis.diagnosticMode": "openFilesOnly",
            "python.analysis.typeCheckingMode": "basic",
            "python.analysis.autoSearchPaths": True,
            "python.analysis.indexing": False,
            "python.analysis.packageIndexDepths": [{"name": "", "depth": 1}],
            "python.analysis.autoImportCompletions": False,
            "python.analysis.completeFunctionParens": False,
            "python.analysis.inlayHints.functionReturnTypes": False,
            "python.analysis.inlayHints.variableTypes": False,
            "python.languageServer": "Pylance",
            "python.analysis.memory.keepLibraryAst": False,
            "python.analysis.exclude": [
                "**/node_modules/**",
                "**/__pycache__/**",
                "**/build/**",
                ".git/**",
                "**/.venv/**",
                "**/env/**",
                "dist/**",
                "o3de/**",
                "unity-integration/**",
                "unity-setup/**",
                "test-results/**",
                "cypress/**",
                "playwright/**"
            ]
        },
        "focused_include": {
            "python.analysis.diagnosticMode": "openFilesOnly",
            "python.analysis.typeCheckingMode": "basic",
            "python.analysis.autoSearchPaths": False,
            "python.analysis.indexing": False,
            "python.analysis.include": [
                "src/**/*.py",
                "backend/**/*.py",
                "components/**/*.py"
            ],
            "python.analysis.exclude": [
                "**/node_modules/**",
                "**/__pycache__/**",
                "**/build/**",
                ".git/**",
                "**/.venv/**",
                "**/env/**",
                "dist/**",
                "o3de/**",
                "unity-integration/**",
                "unity-setup/**",
                "test-results/**",
                "cypress/**",
                "playwright/**"
            ]
        },
        "heavy_indexing": {
            "python.analysis.diagnosticMode": "workspace",
            "python.analysis.typeCheckingMode": "basic",
            "python.analysis.autoSearchPaths": True,
            "python.analysis.indexing": True,
            "python.analysis.packageIndexDepths": [{"name": "", "depth": 2}],
            "python.analysis.autoImportCompletions": True,
            "python.analysis.completeFunctionParens": True,
            "python.analysis.inlayHints.functionReturnTypes": True,
            "python.analysis.inlayHints.variableTypes": True,
            "python.analysis.exclude": [
                "**/node_modules/**",
                "**/__pycache__/**",
                "**/build/**",
                ".git/**"
            ]
        },
        "optimized": optimized_settings
    }

# Function to simulate loading time based on configuration
def simulate_loading_time(config):
    """
    Simulate the loading time of a VS Code configuration
    This is a theoretical model based on configuration complexity
    """
    # Base loading time
    base_time = 5.0
    
    # Factors that increase loading time
    if config.get("python.analysis.diagnosticMode") == "workspace":
        base_time *= 2.0
    
    if config.get("python.analysis.indexing", False):
        base_time *= 1.5
    
    if config.get("python.analysis.autoImportCompletions", False):
        base_time *= 1.2
    
    # Impact of exclude patterns (more is better)
    exclude_patterns = config.get("python.analysis.exclude", [])
    exclude_factor = max(0.5, 1.0 - (len(exclude_patterns) * 0.02))
    base_time *= exclude_factor
    
    # Impact of include patterns (more is worse if not specific enough)
    include_patterns = config.get("python.analysis.include", [])
    if include_patterns:
        # Simplified - assuming more specific patterns
        include_factor = 1.0 + (len(include_patterns) * 0.05)
        base_time *= include_factor
    
    # Add some randomness
    import random
    base_time *= random.uniform(0.9, 1.1)
    
    return base_time

# Let's test our different configurations
test_configs = get_test_configurations()

# Simulate loading times for each configuration
results = []
for name, config in test_configs.items():
    # Run multiple simulations and take the average
    times = []
    for _ in range(5):
        times.append(simulate_loading_time(config))
    
    avg_time = sum(times) / len(times)
    results.append({
        "name": name,
        "avg_loading_time": avg_time,
        "config_complexity": len(json.dumps(config))
    })

# Convert to DataFrame for visualization
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("avg_loading_time")

# Display results
print(results_df)

# Visualize results
plt.figure(figsize=(10, 6))
plt.barh(results_df["name"], results_df["avg_loading_time"], color='skyblue')
plt.xlabel("Simulated Loading Time (seconds)")
plt.title("VS Code Configuration Performance Comparison")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 5. Creating a Custom Workspace Scanner

Finally, let's develop a Python-based workspace scanner that can be used to analyze large workspaces and generate optimized VS Code workspace configurations automatically based on usage patterns.

In [ ]:
class WorkspaceOptimizer:
    """
    A comprehensive workspace analyzer and optimizer for VS Code
    
    Features:
    - Analyzes workspace structure and file types
    - Tracks file access patterns to identify hot spots
    - Generates optimized VS Code settings
    - Creates custom workspace files for multi-root setups
    """
    
    def __init__(self, workspace_path):
        self.workspace_path = workspace_path
        self.stats = None
        self.file_access_history = {}
        self.settings = {}
        self.workspaces = {}
    
    def scan_workspace(self, exclude_patterns=None):
        """Scan the workspace to gather statistics"""
        logger.info(f"Scanning workspace: {self.workspace_path}")
        
        with Timer("Full Workspace Scan"):
            self.stats = scan_workspace(self.workspace_path, exclude_patterns)
        
        return self.stats
    
    def analyze_imports(self):
        """Analyze Python imports to detect dependencies between files"""
        import_patterns = {}
        
        # Find all Python files
        python_files = []
        for root, _, files in os.walk(self.workspace_path):
            for file in files:
                if file.endswith('.py'):
                    python_files.append(os.path.join(root, file))
        
        logger.info(f"Analyzing imports in {len(python_files)} Python files")
        
        # Process Python files to extract imports
        for py_file in tqdm(python_files[:100]):  # Limit to 100 files for demo
            try:
                with open(py_file, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                # Simple regex to find imports
                import_matches = re.finditer(r'^\s*(?:from|import)\s+([a-zA-Z0-9_.]+)', content, re.MULTILINE)
                
                rel_path = os.path.relpath(py_file, self.workspace_path)
                import_patterns[rel_path] = []
                
                for match in import_matches:
                    imported_module = match.group(1).split('.')[0]
                    import_patterns[rel_path].append(imported_module)
            
            except (UnicodeDecodeError, PermissionError, FileNotFoundError) as e:
                logger.warning(f"Error processing file {py_file}: {e}")
        
        return import_patterns
    
    def generate_workspace_file(self, name, folders, settings=None):
        """Generate a VS Code workspace file"""
        if settings is None:
            settings = {}
        
        workspace_content = {
            "folders": [{"path": f} for f in folders],
            "settings": settings
        }
        
        output_path = os.path.join(self.workspace_path, f"{name}.code-workspace")
        with open(output_path, 'w') as f:
            json.dump(workspace_content, f, indent=2)
        
        logger.info(f"Created workspace file: {output_path}")
        return output_path
    
    def create_focused_workspaces(self):
        """Create focused workspaces based on directory structure and file types"""
        # Identify main directories with significant files
        main_dirs = {}
        
        for root, dirs, files in os.walk(self.workspace_path):
            rel_path = os.path.relpath(root, self.workspace_path)
            
            # Skip root
            if rel_path == '.':
                continue
            
            # Skip deeply nested paths
            if len(Path(rel_path).parts) > 2:
                continue
            
            # Count files by type
            type_counts = Counter()
            for file in files:
                _, ext = os.path.splitext(file)
                type_counts[ext.lower()] += 1
            
            # Only consider directories with significant file counts
            if sum(type_counts.values()) > 10:
                main_dirs[rel_path] = type_counts
        
        # Group directories by primary file type
        grouped_dirs = defaultdict(list)
        
        for dir_path, type_counts in main_dirs.items():
            if not type_counts:
                continue
                
            # Get the most common file type
            most_common = type_counts.most_common(1)[0][0]
            
            # Skip if no clear pattern
            if most_common == '':
                continue
                
            grouped_dirs[most_common].append(dir_path)
        
        # Create workspaces for each major file type group
        workspaces = {}
        
        # JavaScript/TypeScript workspace
        js_types = ['.js', '.jsx', '.ts', '.tsx']
        js_dirs = []
        for ext in js_types:
            js_dirs.extend(grouped_dirs.get(ext, []))
        
        if js_dirs:
            workspaces['frontend'] = {
                "folders": js_dirs,
                "settings": {
                    "files.exclude": {
                        "**/node_modules": true,
                        "**/.git": true,
                        "**/dist": true,
                        "**/build": true
                    }
                }
            }
        
        # Python workspace
        py_dirs = grouped_dirs.get('.py', [])
        if py_dirs:
            workspaces['backend'] = {
                "folders": py_dirs,
                "settings": {
                    "python.analysis.diagnosticMode": "openFilesOnly",
                    "python.analysis.exclude": [
                        "**/__pycache__/**",
                        "**/.venv/**",
                        "**/env/**"
                    ]
                }
            }
        
        # Assets workspace (images, etc.)
        asset_types = ['.png', '.jpg', '.jpeg', '.svg', '.gif', '.webp']
        asset_dirs = []
        for ext in asset_types:
            asset_dirs.extend(grouped_dirs.get(ext, []))
        
        if asset_dirs:
            workspaces['assets'] = {
                "folders": asset_dirs,
                "settings": {}
            }
        
        return workspaces
    
    def generate_settings_json(self, optimized=True):
        """Generate a VS Code settings.json file"""
        if optimized and self.stats:
            settings = generate_vscode_settings(self.workspace_path, self.stats)
        else:
            # Default settings with basic optimizations
            settings = {
                "python.analysis.diagnosticMode": "openFilesOnly",
                "python.analysis.typeCheckingMode": "basic",
                "python.analysis.indexing": False,
                "python.analysis.autoImportCompletions": False,
                "files.exclude": {
                    "**/node_modules": True,
                    "**/.git": True,
                    "**/__pycache__": True,
                    "**/dist": True,
                    "**/build": True
                },
                "search.exclude": {
                    "**/node_modules": True,
                    "**/bower_components": True,
                    "**/dist": True,
                    "**/coverage": True
                }
            }
        
        output_path = os.path.join(self.workspace_path, ".vscode", "settings.json")
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        with open(output_path, 'w') as f:
            json.dump(settings, f, indent=2)
        
        logger.info(f"Created settings file: {output_path}")
        return output_path
    
    def create_helper_script(self):
        """Create a helper script to switch between workspaces"""
        script_content = """#!/bin/bash
# VS Code Workspace Selector
# Auto-generated by WorkspaceOptimizer

BLUE='\033[0;34m'
GREEN='\033[0;32m'
YELLOW='\033[1;33m'
NC='\033[0m' # No Color

echo -e "${BLUE}VS Code Workspace Selector${NC}"
echo -e "${YELLOW}Select a workspace to open:${NC}"

# List available workspace files
workspaces=()
i=1
for ws in *.code-workspace; do
    if [ -f "$ws" ]; then
        echo "$i) ${ws%.code-workspace}"
        workspaces+=("$ws")
        ((i++))
    fi
done

# Add option for full workspace
echo "$i) Full workspace (may be slow)"
echo "q) Quit"

read -p "Enter choice [1-$i or q]: " choice

if [[ $choice == "q" ]]; then
    echo -e "${GREEN}Exiting...${NC}"
    exit 0
elif [[ $choice -eq $i ]]; then
    echo -e "${YELLOW}Opening full workspace...${NC}"
    code .
elif [[ $choice -ge 1 && $choice -lt $i ]]; then
    selected=${workspaces[$choice-1]}
    echo -e "${GREEN}Opening ${selected}...${NC}"
    code "$selected"
else
    echo -e "${YELLOW}Invalid choice${NC}"
    exit 1
fi
"""
        
        output_path = os.path.join(self.workspace_path, "open-workspace.sh")
        
        with open(output_path, 'w') as f:
            f.write(script_content)
        
        # Make executable
        os.chmod(output_path, 0o755)
        
        logger.info(f"Created helper script: {output_path}")
        return output_path
    
    def optimize_workspace(self):
        """Run the full optimization process"""
        # 1. Scan the workspace
        if not self.stats:
            self.scan_workspace()
        
        # 2. Create optimized settings
        settings_path = self.generate_settings_json(optimized=True)
        
        # 3. Create focused workspaces
        workspaces = self.create_focused_workspaces()
        
        # 4. Generate workspace files
        workspace_files = []
        for name, ws_config in workspaces.items():
            output_path = self.generate_workspace_file(
                f"{name}-dev",
                ws_config["folders"],
                ws_config.get("settings", {})
            )
            workspace_files.append(output_path)
        
        # 5. Create helper script
        script_path = self.create_helper_script()
        
        return {
            "settings": settings_path,
            "workspaces": workspace_files,
            "helper_script": script_path
        }

In [ ]:
# Create an instance of our WorkspaceOptimizer
optimizer = WorkspaceOptimizer(WORKSPACE_PATH)

# Demonstrate the workspace optimization process
with Timer("Workspace Optimization"):
    optimization_results = optimizer.optimize_workspace()

print("\nOptimization Results:")
for key, value in optimization_results.items():
    print(f"{key}: {value}")

print("\n✅ Done! Your workspace has been optimized for VS Code.")
print("You can now use the helper script to switch between workspaces:")
print("./open-workspace.sh")

## Conclusion

This notebook has demonstrated several techniques for optimizing VS Code performance with large workspaces:

1. **Workspace Analysis**: Understanding your workspace structure and identifying potential performance bottlenecks.

2. **Pylance Configuration**: Using optimal settings for the Python language server to reduce CPU and memory usage:
   - Setting `diagnosticMode` to `openFilesOnly`
   - Disabling `indexing` for large workspaces
   - Using targeted `include` and comprehensive `exclude` patterns
   - Reducing `packageIndexDepths` and other memory-intensive features

3. **Multi-root Workspaces**: Creating focused workspace files that only include the directories you need for specific tasks.

4. **Automation**: Building tools to automatically scan and configure your workspace for optimal performance.

By applying these techniques, you can significantly improve VS Code's performance when working with large codebases.

### Next Steps

- Test the generated workspace files and settings to see which configuration works best for your workflow
- Run the optimizer whenever your project structure changes significantly
- Consider creating specialized workspace files for different development tasks